# SolveX 3.0: Model Training & Evaluation Pipeline

This notebook trains, evaluates, compares, and exports machine learning models for the SolveX emergency detection platform.

### Objectives:
1. **Load Preprocessed Matrices:** Ingest `train_dataset_preprocessed.csv` and `test_dataset_preprocessed.csv`.
2. **Train Benchmark Models:**
   - Model A: **Random Forest Classifier** (ensemble decision trees with balanced class weights)
   - Model B: **Gradient Boosting Classifier** (boosted sequential trees)
3. **Model Evaluation:**
   - Multi-class classification report (Precision, Recall, F1-Score).
   - Confusion matrix visualization across **Code BLUE**, **Code YELLOW**, and **Code RED**.
4. **Feature Importance Ranking:** Identify the strongest determinants of accident severity.
5. **Model Serialization:** Export the trained model to `severity_model.joblib` for deployment in the FastAPI backend.


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

CLEANED_DIR = os.path.join(os.getcwd(), 'data', 'cleaned')
MODEL_EXPORT_PATH = os.path.join(os.path.dirname(os.getcwd()), 'logic', 'severity_model.joblib')
LOCAL_EXPORT_PATH = os.path.join(os.getcwd(), 'severity_model.joblib')

print(f"Cleaned data directory: {CLEANED_DIR}")


## 1. Load Preprocessed Datasets


In [ ]:
train_df = pd.read_csv(os.path.join(CLEANED_DIR, 'train_dataset_preprocessed.csv'))
test_df = pd.read_csv(os.path.join(CLEANED_DIR, 'test_dataset_preprocessed.csv'))

target_cols = ['target_tier', 'target_score']
feature_cols = [c for c in train_df.columns if c not in target_cols]

X_train = train_df[feature_cols]
y_train = train_df['target_tier']

X_test = test_df[feature_cols]
y_test = test_df['target_tier']

# Tier mapping: 0=BLUE, 1=YELLOW, 2=RED
target_names = ['BLUE (Info)', 'YELLOW (Warning)', 'RED (Critical)']

print(f"Train samples: {X_train.shape[0]:,} | Features: {X_train.shape[1]}")
print(f"Test samples:  {X_test.shape[0]:,}  | Features: {X_test.shape[1]}")
print(f"Class distribution in test set:\n{y_test.value_counts(normalize=True).round(3) * 100}")


## 2. Model 1: Random Forest Classifier


In [ ]:
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("=== Random Forest: Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=target_names))


## 3. Model 2: Gradient Boosting Classifier


In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=120,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)

print("=== Gradient Boosting: Classification Report ===")
print(classification_report(y_test, y_pred_gb, target_names=target_names))


## 4. Evaluation & Confusion Matrix Comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest CM
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names, ax=axes[0])
axes[0].set_title(f'Random Forest (Acc: {accuracy_score(y_test, y_pred_rf):.2%})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Tier')
axes[0].set_ylabel('True Tier')

# Gradient Boosting CM
cm_gb = confusion_matrix(y_test, y_pred_gb)
sns.heatmap(cm_gb, annot=True, fmt='d', cmap='Greens', xticklabels=target_names, yticklabels=target_names, ax=axes[1])
axes[1].set_title(f'Gradient Boosting (Acc: {accuracy_score(y_test, y_pred_gb):.2%})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Tier')
axes[1].set_ylabel('True Tier')

plt.tight_layout()
plt.show()


## 5. Feature Importance Analysis


In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False).head(15)

plt.figure(figsize=(11, 6))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Top 15 Most Predictive Features for Emergency Triage', fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance (Gini)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


## 6. Model Serialization & Export

We save the best performing model (`rf`) using `joblib` so it can be directly loaded into the SolveX dispatch pipeline.


In [ ]:
# Save model locally and to logic directory if available
best_model = rf

joblib.dump(best_model, LOCAL_EXPORT_PATH)
print(f"[OK] Saved trained model to {LOCAL_EXPORT_PATH}")

try:
    joblib.dump(best_model, MODEL_EXPORT_PATH)
    print(f"[OK] Exported trained model to backend logic: {MODEL_EXPORT_PATH}")
except Exception as e:
    print(f"Backend export note: {e}")

# Test inference on a sample crash scenario
sample_crash = X_test.iloc[[0]]
predicted_tier = best_model.predict(sample_crash)[0]
predicted_probs = best_model.predict_proba(sample_crash)[0]

print("\n=== Sample Inference Test ===")
print(f"Predicted Tier: {target_names[predicted_tier]}")
for name, prob in zip(target_names, predicted_probs):
    print(f"  • Probability {name:<17}: {prob:.1%}")
